# Multimodal Medical RAG
Text + embedded images + rendered visual pages (diagrams, charts, tables) retrieved together and answered by a vision-capable LLM.

**Install once:**
```
pip install pymupdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-google-genai chromadb sentence-transformers transformers torch pillow numpy
```

### 1. Imports and setup

In [ ]:
import os
import base64
import getpass
import warnings
from io import BytesIO
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import torch
import chromadb
from PIL import Image

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import CLIPProcessor, CLIPModel


DATA_PATH = Path(r"\Users\avinash\Desktop\medical-evidence-rag\data")             # folder with your PDFs
IMAGE_DIR = Path("extracted_images")  # extracted images + rendered pages are saved here
IMAGE_DIR.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("PDFs found:", [p.name for p in DATA_PATH.glob("*.pdf")])
   

Device: cpu
PDFs found: ['ilovepdf_merged (1).pdf']


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

### 2. Extract text, images and visual pages from every PDF

In [31]:
from transformers import CLIPProcessor, CLIPModel

# load CLIP here because we use it to check if a picture is a chart
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

labels = ["a bar chart", "a pie chart", "a line graph", "a flowchart diagram",
          "a photograph", "a logo", "a table", "a page of text"]

documents = []      # text
image_data = []     # images
visual_pages = []   # charts / graphs

for pdf_file in DATA_PATH.glob("*.pdf"):
    pdf_doc = fitz.open(str(pdf_file))

    for page_number, page in enumerate(pdf_doc):

        # ---------- 1. TEXT ----------
        text = page.get_text().strip()
        if text:
            documents.append(Document(
                page_content=text,
                metadata={"source": pdf_file.name, "page": page_number + 1}
            ))

        # ---------- 2. IMAGES ----------
        for image_index, image in enumerate(page.get_images(full=True)):
            image_bytes = pdf_doc.extract_image(image[0])

            if image_bytes["width"] < 100 or image_bytes["height"] < 100:
                continue    # skip small logos and icons

            path = IMAGE_DIR / f"{pdf_file.stem}_p{page_number + 1}_img{image_index + 1}.{image_bytes['ext']}"
            path.write_bytes(image_bytes["image"])

            image_data.append({
                "content_type": "image",
                "source": pdf_file.name,
                "page": page_number + 1,
                "image_index": image_index + 1,
                "image": image_bytes["image"],
                "extension": image_bytes["ext"],
                "path": str(path)
            })

        # ---------- 3. VISUAL IMAGES (charts / graphs) ----------
        areas = [fitz.Rect(info["bbox"]) for info in page.get_image_info()]
        areas += page.cluster_drawings()

        for area in areas:
            if area.width < 150 or area.height < 100:
                continue    # skip small things

            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), clip=area)
            image = Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")

            inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
            probs = clip_model(**inputs).logits_per_image.softmax(dim=1)[0]
            label = labels[probs.argmax().item()]

            if label in labels[:4]:    # first 4 labels are charts
                path = IMAGE_DIR / f"{pdf_file.stem}_p{page_number + 1}_chart{len(visual_pages)}.png"
                image.save(path)

                visual_pages.append({
                    "content_type": "visual",
                    "source": pdf_file.name,
                    "page": page_number + 1,
                    "chart_type": label,
                    "image": pix.tobytes("png"),
                    "path": str(path)
                })

    pdf_doc.close()

print("Text pages:", len(documents))
print("Images:", len(image_data))
print("Visual pages:", len(visual_pages))
for v in visual_pages:
    print("Page", v["page"], "->", v["chart_type"])

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Text pages: 57
Images: 8
Visual pages: 12
Page 7 -> a flowchart diagram
Page 7 -> a flowchart diagram
Page 11 -> a flowchart diagram
Page 14 -> a bar chart
Page 15 -> a bar chart
Page 15 -> a bar chart
Page 15 -> a bar chart
Page 16 -> a bar chart
Page 16 -> a line graph
Page 17 -> a bar chart
Page 17 -> a line graph
Page 17 -> a line graph


### 3. Text chunking

In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents) 

print("Total documents:", len(documents))
print("Total chunks   :", len(chunks))

Total documents: 57
Total chunks   : 381


### 4. Inspect metadata

In [12]:
for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i + 1}:", chunk.metadata)

Chunk 1: {'source': 'ilovepdf_merged (1).pdf', 'page': 1}
Chunk 2: {'source': 'ilovepdf_merged (1).pdf', 'page': 1}
Chunk 3: {'source': 'ilovepdf_merged (1).pdf', 'page': 1}
Chunk 4: {'source': 'ilovepdf_merged (1).pdf', 'page': 1}
Chunk 5: {'source': 'ilovepdf_merged (1).pdf', 'page': 1}


### 5. Load embedding models
- Text: `all-MiniLM-L6-v2` (384-dim)
- Images / visual pages: CLIP (512-dim)

In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)
print("Text embedding model loaded")

CLIP_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(device)
clip_model.eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
print("CLIP model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Text embedding model loaded


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model loaded


### 6. CLIP helper functions

In [43]:
def _to_tensor(output):
    """Newer transformers versions may return an output object instead of a tensor."""
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds
    return output.pooler_output


def embed_image(image_bytes):
    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        vector = _to_tensor(clip_model.get_image_features(**inputs))
    vector = vector / vector.norm(dim=-1, keepdim=True)  # normalise for cosine similarity
    return vector.cpu().numpy()[0]


def embed_clip_text(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        vector = _to_tensor(clip_model.get_text_features(**inputs))
    vector = vector / vector.norm(dim=-1, keepdim=True)
    return vector.cpu().numpy()[0]

### 7. Generate image and visual-page embeddings

In [44]:
image_embeddings = []
for item in image_data:
    image_embeddings.append(embed_image(item["image"]))

visual_embeddings = []
for item in visual_pages:
    visual_embeddings.append(embed_image(item["image"]))

print("Image embeddings :", len(image_embeddings))
print("Visual embeddings:", len(visual_embeddings))
if image_embeddings:
    print("CLIP dimension   :", image_embeddings[0].shape)

Image embeddings : 0
Visual embeddings: 12


### 8. Store in Chroma
Text (384-dim) and images (512-dim) have different dimensions, so they **must** go into separate collections.

In [16]:
# ---- Text collection (LangChain wrapper) ----
text_vectorstore = Chroma(
    collection_name="text_documents",
    embedding_function=embeddings,
    persist_directory="./text_db",
    collection_metadata={"hnsw:space": "cosine"},
)
text_vectorstore.reset_collection()  # avoid duplicates when re-running the cell
text_vectorstore.add_documents(
    chunks,
    ids=[f"text_{i}" for i in range(len(chunks))],
)
print("Text chunks stored:", text_vectorstore._collection.count())

# ---- Image + visual-page collection (raw chromadb, CLIP vectors) ----
chroma_client = chromadb.PersistentClient(path="./image_db")
try:
    chroma_client.delete_collection("image_documents")
except Exception:
    pass
image_collection = chroma_client.create_collection(
    name="image_documents",
    metadata={"hnsw:space": "cosine"},
)

if image_embeddings:
    image_collection.add(
        ids=[f"image_{i}" for i in range(len(image_embeddings))],
        embeddings=[v.tolist() for v in image_embeddings],
        metadatas=[
            {
                "content_type": "image",
                "source": item["source"],
                "page": item["page"],
                "image_index": item["image_index"],
                "path": item["path"],
            }
            for item in image_data
        ],
    )

if visual_embeddings:
    image_collection.add(
        ids=[f"visual_{i}" for i in range(len(visual_embeddings))],
        embeddings=[v.tolist() for v in visual_embeddings],
        metadatas=[
            {
                "content_type": "visual",
                "source": item["source"],
                "page": item["page"],
                "path": item["path"],
            }
            for item in visual_pages
        ],
    )

print("Image + visual vectors stored:", image_collection.count())

Text chunks stored: 381
Image + visual vectors stored: 20


### 9. Retriever functions

In [47]:
retriever = text_vectorstore.as_retriever(search_kwargs={"k": 5})


def retrieve_text(query, k=5):
    return text_vectorstore.similarity_search(query, k=k)


def retrieve_visuals(query, top_k=3, content_type=None):
    """Search images and/or visual pages with a text query via CLIP.
    content_type: None (both), "image", or "visual"."""
    if image_collection.count() == 0:
        return []
    query_vector = embed_clip_text(query)
    results = image_collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=min(top_k, image_collection.count()),
        where={"content_type": content_type} if content_type else None,
    )
    hits = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        hits.append({**meta, "score": round(1 - dist, 4)})
    return hits


query = "What is the proposed system architecture?"

for doc in retrieve_text(query):
    print("\nPage:", doc.metadata.get("page"), "| Source:", doc.metadata.get("source"))
    print(doc.page_content[:300])

print("\nTop images/visual pages:")
for hit in retrieve_visuals(query, top_k=3):
    print(hit)


Page: 13 | Source: ilovepdf_merged (1).pdf
Conceptual: Studies that propose a conceptual digital system architecture without a tested 
application.  
Simulation: Studies in which a system is designed and tested locally but not converted into a real-time 
application.  
Decentralized Application (DApp): Studies in which the created system can

Page: 45 | Source: ilovepdf_merged (1).pdf
online:  
https:  
//www.fon.hum.uva.nl/rob/Courses/InformationInSpeech/CDROM/Literature/LOTwinterschool2006/sza 
bo.best.vwh. net/universal.html (accessed on 21 November 2022).  
21. Buterin, V. Ethereum: A Next-Generation Smart Contract and Decentralized Application Platform.  
 
Home|ethereum.org

Page: 7 | Source: ilovepdf_merged (1).pdf
Blockchain-Enabled ML for Water Quality & Billing                                                        DEPT.OF.CSE                             
KOMMURI PRATAP REDDY INSTITUTE OF TECHNOLOGY                                              Page 7   
   
   
   
Fig. 1: P

### 10. Load the LLM
Uses Google Gemini (vision-capable, so it can read the retrieved images). The key is read from the `GOOGLE_API_KEY` environment variable, or you'll be prompted — don't hard-code it in the notebook.

In [ ]:
import os
api_key=os.environ["OPENAI_API_KEY"]="YOUR_API_KEY"
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-5.6-luna")
print(llm.invoke("Reply with OK").content)

OK


### 11. Text-only RAG chain

In [25]:
def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source')} - page {d.metadata.get('page')}]\n{d.page_content}"
        for d in docs
    )


prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}
""")

text_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(text_chain.invoke("What is the proposed system architecture?"))

The provided context does not describe the proposed system architecture; it only references “Fig. 1: Proposed system architecture” and lists its software and hardware components.


### 12. Multimodal RAG (text + images + visual pages)

In [ ]:
def image_to_data_url(path):
    ext = Path(path).suffix.lstrip(".").lower()
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/{mime};base64,{encoded}"


def multimodal_rag(question, k_text=5, k_images=2, k_visual=2):
    text_docs = retrieve_text(question, k=k_text)
    image_hits = retrieve_visuals(question, top_k=k_images, content_type="image")
    visual_hits = retrieve_visuals(question, top_k=k_visual, content_type="visual")
    all_visuals = image_hits + visual_hits

    content = [{
        "type": "text",
        "text": (
            "You are a document assistant. Answer the question using ONLY the "
            "text context and the attached images (extracted figures and full page renders). "
            "Cite page numbers. If the answer is not present, say you don't know.\n\n"
            f"Text context:\n{format_docs(text_docs)}\n\n"
            f"Question: {question}"
        ),
    }]

    for hit in all_visuals:
        content.append({
            "type": "text",
            "text": f"[{hit['content_type']} from {hit['source']} page {hit['page']}]",
        })
        content.append({
            "type": "image_url",
            "image_url": {"url": image_to_data_url(hit["path"])},
        })

    response = llm.invoke([HumanMessage(content=content)])
    return {
        "answer": response.content,
        "text_sources": [(d.metadata.get("source"), d.metadata.get("page")) for d in text_docs],
        "visual_sources": all_visuals,
    }


result = multimodal_rag("What is the proposed system architecture?")
print(result["answer"])
print("\nText sources:", result["text_sources"])
print("Visual sources:")
for v in result["visual_sources"]:
    print("  ", v["content_type"], "| page", v["page"], "| score", v["score"], "|", v["path"])

The proposed system architecture is a **three-tier blockchain-enabled machine-learning system** for water-quality prediction and tamper-proof billing:

- **Presentation tier:** A Django web application server provides the user interface.
- **Application/logic tier:** A Python Web3 client performs data preprocessing using Pandas/NumPy, synchronizes data, and runs machine-learning models using Scikit-learn/Random Forest. The system produces water-quality predictions and supports tamper-proof allocation and billing transactions.
- **Data tier:** The system uses a Ganache/Ethereum blockchain node with Solidity smart contracts, alongside storage for the ML model (`model/rf.pkl`) and the water-potability dataset (`dataset/water_potability.csv`). The architecture is intended to provide immutable data access. **(p. 7)**

In the blockchain transaction flow, participants use public/private keys to interact with the smart contract. Update transactions are validated and recorded in the distributed

### 13. Show the retrieved images

In [ ]:
from IPython.display import display

for v in result["visual_sources"]:
    print(f"{v['content_type']} - {v['source']} page {v['page']} (score {v['score']})")
    img = Image.open(v["path"])
    img.thumbnail((700, 700))
    display(img)

### 14. Ask your own questions

In [ ]:
while True:
    q = input("\nAsk a question (or 'exit'): ").strip()
    if q.lower() in ("exit", "quit", ""):
        break
    out = multimodal_rag(q)
    print("\nAnswer:\n", out["answer"])
   


Answer:
 The proposed system introduced a **blockchain-enabled machine-learning architecture for water-quality prediction and billing**. Its main components are:

- **Presentation tier:** A Django-based web application.
- **Application/logic tier:** A Python Web3 client, data preprocessing using Pandas/NumPy, data synchronization, and a machine-learning module using Scikit-learn/Random Forest for water-quality prediction.
- **Blockchain functions:** Immutable data access and tamper-proof allocation and billing transactions through a blockchain node using Ganache/Ethereum and a Solidity smart contract.
- **Data tier:** Storage for the trained ML model and the water-potability dataset.  
  *(Page 7)*

The related blockchain workflow also shows participants using public/private keys to interact with a smart contract; transactions are validated and recorded on a distributed ledger. *(Page 11)*
Pages used: [8, 13, 23, 49, 55]
